### Data Ingestion

In [ ]:
import os
from pathlib import Path

work_dir = os.getenv("WORK_DIR")
os.makedirs(f"{work_dir}/knowledge/text",exist_ok=True)

In [ ]:
from langchain_core.documents import Document

doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"exmaple.txt",
        "pages":1,
        "author":"Brijesh K Dhaker",
        "date_created":"2025-01-01"
    }
)
doc

Document(metadata={'source': 'exmaple.txt', 'pages': 1, 'author': 'Krish Naik', 'date_created': '2025-01-01'}, page_content='this is the main text content I am using to create RAG')

In [ ]:
sample_texts={
    "knowledge/text/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "knowledge/text/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(f"{work_dir}/{filepath}",'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [5]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader=TextLoader(f"{work_dir}/knowledge/text/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

[Document(metadata={'source': '/home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/text/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [6]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    f"{work_dir}/knowledge/text",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False
)

documents=dir_loader.load()
documents

[Document(metadata={'source': '/home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/text/designs.txt'}, page_content='Monolithic vs. Microservices Architecture\nWith monolithic architectures, all processes are tightly coupled and run as a single service. This means that if one process of the application experiences a spike in demand, the entire architecture must be scaled. Adding or improving a monolithic application’s features becomes more complex as the code base grows. This complexity limits experimentation and makes it difficult to implement new ideas. Monolithic architectures add risk for application availability because many dependent and tightly coupled processes increase the impact of a single process failure.\nWith a microservices architecture, an application is built as independent components that run each application process as a service. These services communicate via a well-defined interface using lightweight APIs. Services are built for business capabilities and ea

In [8]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    f"{work_dir}/knowledge/pdfs",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False
)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2024-01-19T08:57:33+05:30', 'source': '/home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/pdfs/esop_info.pdf', 'file_path': '/home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/pdfs/esop_info.pdf', 'total_pages': 9, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-01-19T08:57:39+05:30', 'trapped': '', 'modDate': "D:20240119085739+05'30'", 'creationDate': "D:20240119085733+05'30'", 'page': 0}, page_content='CS Ashwani Singh Bisht, ACS\nCompany Secretary & Compliance Officer\nBartronics India Limited\nMadhapur, Hyderabad\nashwanisingh900@gmail.com\nEmployee Stock Option Plan (ESOP) : The Finer \nNuances\nAn Employee Stock Option Plan (ESOP) is an employee benefit plan that gives workers ownership \ninterest in the company in the form of shares or stock of the company. One thing we should keep in \nmind that

In [9]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [3]:
### Read all the documents inside the directory
import os
from com.example.ai.loader.LoadManager import LoadManager

json_documents = LoadManager.from_directory(f"{os.getenv("WORK_DIR")}/knowledge", inclusions=["json"])
print(f"[*INFO] Total loaded documents: {len(json_documents)}")

[DEBUG] Data path: /home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge
[DEBUG] Found 2 page : ['/home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/json/articals.json', '/home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/json/designs-research.json']
[DEBUG] Loading json from : /home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/json/articals.json
[DEBUG] Loaded 10 JSON document from /home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/json/articals.json
[DEBUG] Loading json from : /home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/json/designs-research.json
[ERROR] Failed to load JSON document /home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/json/designs-research.json: Cannot iterate over null (null)
[DEBUG] Total loaded documents: 10
[*INFO] Total loaded documents: 10


In [4]:
### Text splitting get into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

_chunks = splitter.split_documents(json_documents)
print(f"[INFO] Split {len(json_documents)} documents into {len(_chunks)} chunks.")
# Show example of a chunk
if _chunks:
    print(f"Sample Chunk [0]:")
    print(f"Content: {_chunks[0].page_content[:200]}...")
    print(f"Metadata: {_chunks[0].metadata}")

[INFO] Split 10 documents into 10 chunks.
Sample Chunk [0]:
Content: The report adds to the growing documentation on commercial data’s contributions to Earth science research and applications....
Metadata: {'source': '/home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/json/articals.json', 'seq_num': 1}


### VectorStore

In [5]:
from com.example.ai.vectors.VectorStoreManager import VectorStoreManager

vectorStoreManager = VectorStoreManager(store_type="faissdb", collectionOrIndexName="faiss_index")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Add Documents to Vectorstore

In [ ]:
vectorStoreManager.add_documents(json_documents)

### Retriever Pipeline From VectorStore

In [6]:
#
ctx_retriever = vectorStoreManager.retriever(search_type="similarity")
ctx_retriever


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x71311c25c440>, search_kwargs={})

In [7]:
ctx_retriever.invoke("All recipes with rice ?")

[Document(id='e2d5839c-4b05-4f8f-a0c9-555b6b83f8b1', metadata={'producer': 'Mac OS X 10.6.4 Quartz PDFContext', 'creator': 'Microsoft Word', 'creationdate': "D:20100809161642Z00'00'", 'title': 'Easy recipes', 'author': 'sylvie meynier', 'subject': '', 'moddate': "D:20100809161642Z00'00'", 'keywords': '', 'aapl:keywords': "['']", 'source': '/home/brijeshdhaker/IdeaProjects/bd-crewai-module/knowledge/pdfs/Easy_recipes.pdf', 'total_pages': 15, 'page': 7, 'page_label': '8'}, page_content='6. Vegetarian\t\r \xa0Rice\t\r \xa0Ingredients\t\r \xa01/2\t\r \xa0chopped\t\r \xa0onion\t\r \xa01\t\r \xa0chopped\t\r \xa0garlic\t\r \xa0clove\t\r \xa01\t\r \xa0chopped\t\r \xa0courgette\t\r \xa01\t\r \xa0grated\t\r \xa0carrot\t\r \xa01\t\r \xa0chopped\t\r \xa0red\t\r \xa0or\t\r \xa0yellow\t\r \xa0pepper\t\r \xa075g\t\r \xa0frozen\t\r \xa0peas\t\r \xa0\t\r \xa01/2\t\r \xa0tin\t\r \xa0of\t\r \xa0chopped\t\r \xa0tomatoes\t\r \xa0350ml\t\r \xa0Vegetable\t\r \xa0stock\t\r \xa0150g\t\r \xa0Dried\t\r \xa0brown